# SNI-21 A0/A1/A2 setup — no training

Notebook ini hanya menyiapkan A0 real serta A1/A2 sintetis dan menjalankan audit. **Tidak ada training dan test tidak dibaca.**

- Hanya ingin smoke/audit: runtime CPU cukup.
- Ingin langsung training setelah full setup pada sesi yang sama: pilih GPU **sebelum cell pertama**. Setup sendiri tetap memakai CPU, tetapi `/content` tidak hilang karena runtime tidak diganti di tengah jalan.
- Dataset/output besar ditulis ke `/content`, bukan di-upload dari komputer lokal.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import importlib
import os
import subprocess
import sys

REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch',
                    'https://github.com/ediprin/coffee-bean-detection.git', str(REPO)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
importlib.invalidate_caches()
import coffee_detector
os.chdir(REPO)
print('IMPORT BERHASIL:', coffee_detector.__file__)
print('TRAINING BELUM DIJALANKAN')

In [ ]:
DRIVE = Path('/content/drive/MyDrive')
RAW_ARCHIVES = {
    'adrian_detection': DRIVE / 'coffee-sni-detection-fullscene-v1/adrian_detection.tar',
    'faruq_segmentation': DRIVE / 'coffee-sni-detection-fullscene-v1/faruq_segmentation.tar',
}
CROP_ROOT = DRIVE / 'coffee-sni-instance-crop-v1'
RAW_ROOT = Path('/content/sni21-raw')
A0_ROOT = Path('/content/sni21-fullscene-v1')
SHARD_CACHE = Path('/content/sni21-shard-cache')
SHARED_LIBRARY = Path('/content/sni21-object-library')

for name, path in RAW_ARCHIVES.items():
    assert path.is_file(), f'Arsip {name} tidak ditemukan: {path}'
for required in ('manifest.csv', 'audit.json', 'complete.json'):
    assert (CROP_ROOT / required).is_file(), f'Crop package belum lengkap: {CROP_ROOT / required}'
assert (CROP_ROOT / 'shards').is_dir(), f'Shard crop tidak ditemukan: {CROP_ROOT / "shards"}'
print('SEMUA INPUT DRIVE DITEMUKAN')
print('Tidak ada file yang di-upload dari komputer lokal.')

In [ ]:
import tarfile
import time

def extract_with_progress(archive: Path, target: Path) -> None:
    marker = target / '.extract_complete'
    if marker.is_file():
        print('REUSE EXTRACT:', target)
        return
    if target.exists() and any(target.iterdir()):
        raise RuntimeError(f'Ekstraksi parsial ditemukan; hapus folder ini lalu ulangi: {target}')
    target.mkdir(parents=True, exist_ok=True)
    with tarfile.open(archive, 'r') as bundle:
        members = bundle.getmembers()
        print(f'EXTRACT {archive.name}: 0/{len(members)}', flush=True)
        started = time.perf_counter()
        for index, member in enumerate(members, 1):
            destination = (target / member.name).resolve()
            if destination != target.resolve() and target.resolve() not in destination.parents:
                raise RuntimeError(f'Path TAR tidak aman: {member.name}')
            try:
                bundle.extract(member, target, filter='data')
            except TypeError:
                bundle.extract(member, target)
            if index % 500 == 0 or index == len(members):
                elapsed = time.perf_counter() - started
                rate = index / max(elapsed, 1e-8)
                eta = (len(members) - index) / max(rate, 1e-8)
                print(f'  {index}/{len(members)} | ETA {eta / 60:.1f} menit', flush=True)
    marker.write_text('complete', encoding='utf-8')

def find_coco_root(root: Path) -> Path:
    candidates = [root, *sorted(path for path in root.rglob('*') if path.is_dir())]
    for candidate in candidates:
        if all((candidate / split / '_annotations.coco.json').is_file()
               for split in ('train', 'valid', 'test')):
            return candidate
    raise FileNotFoundError(f'Root COCO train/valid/test tidak ditemukan di {root}')

for name, archive in RAW_ARCHIVES.items():
    extract_with_progress(archive, RAW_ROOT / name)

ADRIAN_ROOT = find_coco_root(RAW_ROOT / 'adrian_detection')
FARUQ_ROOT = find_coco_root(RAW_ROOT / 'faruq_segmentation')
print('ADRIAN:', ADRIAN_ROOT)
print('FARUQ :', FARUQ_ROOT)

In [ ]:
from coffee_detector.audit_dataset import audit_dataset
from coffee_detector.prepare_sni_fullscene import prepare_sni_fullscene

if not (A0_ROOT / 'audit.json').is_file():
    prepare_sni_fullscene(
        ADRIAN_ROOT,
        FARUQ_ROOT,
        CROP_ROOT / 'manifest.csv',
        A0_ROOT,
        seed=42,
        link_mode='hardlink',
    )
else:
    print('REUSE A0:', A0_ROOT)

post = audit_dataset(
    A0_ROOT,
    A0_ROOT / 'post_materialization_audit.json',
    near_threshold=-1,
)
assert post['safe_for_training'], A0_ROOT / 'post_materialization_audit.json'
assert post['cross_split_duplicate_components'] == 0
print('A0 SIAP:', post['images'])
print('TEST TETAP TERKUNCI. TRAINING BELUM DIJALANKAN.')

## Smoke setup

Cell ini membuat object library sekali, lalu hanya 2 scene A1 dan 2 scene A2. Tetap memakai 220–300 objek agar kepadatan sebenarnya diuji. Jalankan ini sebelum full setup.

In [ ]:
from coffee_detector.run_sni21_vadcp_setup import run_sni21_vadcp_setup

SMOKE_ROOT = Path('/content/sni21-vadcp-smoke')
smoke = run_sni21_vadcp_setup(
    A0_ROOT,
    CROP_ROOT,
    SMOKE_ROOT,
    synthetic_images=2,
    seed=42,
    objects_min=220,
    objects_max=300,
    canvas_size=1024,
    shard_cache_root=SHARD_CACHE,
    object_library_root=SHARED_LIBRARY,
    visual_samples=2,
)
print('SMOKE TRAINING_READY =', smoke['training_ready'])
assert smoke['training_ready'], smoke['arms']

In [ ]:
from IPython.display import Image as DisplayImage, display

for arm in ('A1', 'A2'):
    print('\n', arm, 'RAW')
    display(DisplayImage(filename=smoke['arms'][arm]['raw_contact_sheet'], width=900))
    print(arm, 'OVERLAY')
    display(DisplayImage(filename=smoke['arms'][arm]['overlay_contact_sheet'], width=900))

## Full setup — masih bukan training

Jalankan hanya jika smoke terlihat benar. Ubah `RUN_FULL` menjadi `True`. Cell menampilkan progres dan ETA. Hasil tetap berada di `/content`; jangan ganti runtime setelah selesai jika ingin langsung menjalankan training pada sesi yang sama.

In [ ]:
RUN_FULL = False  # UBAH MENJADI True SETELAH SMOKE DISETUJUI

if not RUN_FULL:
    print('FULL SETUP BELUM DIJALANKAN. Tidak ada training.')
else:
    FULL_ROOT = Path('/content/sni21-vadcp-full')
    full = run_sni21_vadcp_setup(
        A0_ROOT,
        CROP_ROOT,
        FULL_ROOT,
        synthetic_images=2000,
        seed=42,
        objects_min=220,
        objects_max=300,
        canvas_size=1024,
        shard_cache_root=SHARD_CACHE,
        object_library_root=SHARED_LIBRARY,
        visual_samples=12,
    )
    print('FULL TRAINING_READY =', full['training_ready'])
    assert full['training_ready'], full['arms']
    print('SETUP SELESAI. TRAINING MASIH BELUM DIJALANKAN.')